In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import datetime
import requests

pd.set_option('display.max_columns', None)
print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## Definición de Funciones
#### Lógica para obtener la lista de empresas del S&P 500 y las fórmulas para las 35 parámetros

In [2]:
def obtener_tickers_sp500():
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        }
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        tables = pd.read_html(response.text)
        df_wiki = tables[0]
        tickers = df_wiki['Symbol'].str.replace('.', '-', regex=False).tolist()
        return tickers
    except Exception as e:
        print(f"Error al obtener tickers: {e}")
        return []

def calcular_35_indicadores(df):
    data = df.copy()

    # --- GRUPO 1: DATOS BASICOS

    # --- GRUPO 2: TENDENCIA Y MEDIAS MOVILES -
    data['SMA_10']  = data['Adj Close'].rolling(window=10).mean()
    data['SMA_50']  = data['Adj Close'].rolling(window=50).mean()
    data['SMA_200'] = data['Adj Close'].rolling(window=200).mean()
    data['EMA_12']  = data['Adj Close'].ewm(span=12, adjust=False).mean()
    data['EMA_26']  = data['Adj Close'].ewm(span=26, adjust=False).mean()

    # --- GRUPO 3: VOLATILIDAD -
    data['Std_20'] = data['Adj Close'].rolling(window=20).std()
    data['Bollinger_Upper'] = data['SMA_50'] + (data['Std_20'] * 2) # Usando SMA local o 20
    data['Bollinger_Lower'] = data['SMA_50'] - (data['Std_20'] * 2)
    data['Daily_Range'] = data['High'] - data['Low']
    data['Range_Pct'] = (data['Daily_Range'] / data['Open']) * 100

    # --- GRUPO 4: RETORNOS -
    data['Daily_Return'] = data['Adj Close'].pct_change()
    data['Log_Return'] = np.log(data['Adj Close'] / data['Adj Close'].shift(1))
    data['Cumulative_Return'] = (1 + data['Daily_Return']).cumprod()
    data['Is_Positive_Day'] = (data['Close'] > data['Open']).astype(int)

    # --- GRUPO 5: MOMENTUM & MACD -
    data['MACD_Line'] = data['EMA_12'] - data['EMA_26']
    data['MACD_Signal'] = data['MACD_Line'].ewm(span=9, adjust=False).mean()
    data['MACD_Hist'] = data['MACD_Line'] - data['MACD_Signal']

    # RSI (Relative Strength Index)
    delta = data['Adj Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    data['RSI_14'] = 100 - (100 / (1 + rs))
    data['Momentum_10'] = data['Adj Close'] - data['Adj Close'].shift(10)

    # --- GRUPO 6: VOLUMEN RELATIVO -
    data['Vol_SMA_20'] = data['Volume'].rolling(window=20).mean()
    data['Vol_Ratio'] = data['Volume'] / data['Vol_SMA_20']
    data['Vol_Change'] = data['Volume'].pct_change()

    # --- GRUPO 7: ESTADÍSTICOS Y SPREADS -
    data['High_Low_Spread'] = (data['High'] - data['Low']) / data['Low']
    data['Close_Open_Spread'] = (data['Close'] - data['Open']) / data['Open']
    data['Typical_Price'] = (data['High'] + data['Low'] + data['Close']) / 3
    data['Z_Score_20'] = (data['Adj Close'] - data['Adj Close'].rolling(20).mean()) / data['Std_20']
    data['Above_SMA200'] = (data['Adj Close'] > data['SMA_200']).astype(int)


    data['Pct_From_High_50'] = (data['Adj Close'] / data['Adj Close'].rolling(50).max()) - 1

    return data

## Descarga de datos (S&P 500)
#### Descarga de data másiva

In [3]:
import os
import pandas as pd
import yfinance as yf
import datetime
from sqlalchemy import create_engine, inspect

# --- CONFIGURACIÓN ---
db_name = 'sp500_market_data.db'
table_name = 'sp500_daily_metrics'
engine = create_engine(f'sqlite:///{db_name}')

# Bandera de control
data_found = False

# 1. VERIFICACIÓN: ¿Existen los datos en SQL?
if os.path.exists(db_name):
    inspector = inspect(engine)
    if table_name in inspector.get_table_names():
        print(f"✅ Base de datos encontrada: {db_name}")
        print(f"   -> Cargando datos desde SQL (omitir descarga)...")

        # Carga rápida a memoria desde el archivo local
        final_df = pd.read_sql(f"SELECT * FROM {table_name}", con=engine)

        data_found = True
        print(f"   -> Carga completada. Filas: {len(final_df):,} | Columnas: {len(final_df.columns)}")

# 2. ACCIÓN: Descargar SOLO si NO se encontraron datos (data_found == False)
if not data_found:
    print(f"⚠️ No se encontraron datos previos. Iniciando proceso de descarga...")

    # A. Obtener Tickers (Asegúrate de tener la función obtener_tickers_sp500 definida previamente)
    tickers = obtener_tickers_sp500()
    print(f"   -> Tickers identificados: {len(tickers)}")

    if len(tickers) == 0:
         raise ValueError("No se pudieron obtener tickers. Revisa la conexión o Wikipedia.")

    # B. Descargar (Últimos 2 años)
    end_date = datetime.datetime.now()
    start_date = end_date - datetime.timedelta(days=730)
    print(f"   -> Descargando desde Yahoo Finance ({start_date.date()} - {end_date.date()})...")

    # Descarga masiva
    raw_data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=False, threads=True)

    # C. Procesar y Calcular (35 Columnas)
    processed_frames = []
    print(f"   -> Calculando indicadores para cada empresa...")

    for ticker in tickers:
        try:
            # Verificamos si el ticker existe en las columnas descargadas
            if ticker not in raw_data.columns.levels[0]:
                continue

            df_ticker = raw_data[ticker].copy()

            # Filtros de calidad de data
            if df_ticker.empty or len(df_ticker) < 50:
                continue

            # Usamos la función de cálculo definida en celdas anteriores
            df_ticker = calcular_35_indicadores(df_ticker)

            # Añadir columna identificadora
            df_ticker['Ticker'] = ticker

            # Limpiar vacíos iniciales (por las medias móviles)
            df_ticker.dropna(inplace=True)

            processed_frames.append(df_ticker)
        except Exception as e:
            continue

    # Concatenar todos los dataframes
    final_df = pd.concat(processed_frames)

    # Reordenar columnas para que Ticker sea la primera
    cols = ['Ticker'] + [c for c in final_df.columns if c != 'Ticker']
    final_df = final_df[cols]

    # D. Guardar en SQL para el futuro
    print(f"   -> Guardando en {db_name}...")

    # Reset index para que la fecha (Date) se guarde como columna explícita
    if 'Date' not in final_df.columns:
        final_df_sql = final_df.reset_index()
    else:
        final_df_sql = final_df.copy()

    final_df_sql.to_sql(name=table_name, con=engine, if_exists='replace', index=False, chunksize=1000)
    print(f"🚀 Proceso finalizado. Datos guardados exitosamente en SQL.")


✅ Base de datos encontrada: sp500_market_data.db
   -> Cargando datos desde SQL (omitir descarga)...
   -> Carga completada. Filas: 151,258 | Columnas: 36


## Guardar en Base de Datos SQL
#### Guardamos toda la data en una tabla de SQL

In [4]:
import os
import pandas as pd
from sqlalchemy import create_engine

db_name = 'sp500_market_data.db'
table_name = 'sp500_daily_metrics'

# Verificamos si el archivo ya existe en el sistema
if os.path.exists(db_name):
    print(f"✅ La base de datos '{db_name}' YA EXISTE.")
    print("No se ha realizado ninguna operación de escritura nueva.")

    engine = create_engine(f'sqlite:///{db_name}')
    try:
        # Hacemos una consulta rápida para confirmar
        with engine.connect() as conn:
            # Esta línea verifica si la tabla existe sin leer toda la data
            check = pd.read_sql(f"SELECT COUNT(*) FROM {table_name}", conn)
        print(f"   -> Conexión exitosa. La tabla '{table_name}' contiene datos.")
    except Exception as e:
        print(f"   -> El archivo existe pero hubo un error al leer: {e}")

else:
    print(f"⚠️ La base de datos '{db_name}' NO existe. Iniciando guardado...")

    # Crear el motor
    engine = create_engine(f'sqlite:///{db_name}')

    # Preparar el DataFrame (reset index para guardar la fecha)
    if 'Date' not in final_df.columns:
        final_df_sql = final_df.reset_index()
    else:
        final_df_sql = final_df.copy()

    # Guardar en SQL
    final_df_sql.to_sql(
        name=table_name,
        con=engine,
        if_exists='replace',
        index=False,
        chunksize=1000
    )

    print(f"🚀 ¡Éxito! Nueva base de datos creada: {db_name}")

✅ La base de datos 'sp500_market_data.db' YA EXISTE.
No se ha realizado ninguna operación de escritura nueva.
   -> Conexión exitosa. La tabla 'sp500_daily_metrics' contiene datos.


## Verificación de SQL

In [5]:

query = """
SELECT Date, Ticker, "Adj Close", RSI_14
FROM sp500_daily_metrics
WHERE Ticker = 'AAPL'
ORDER BY Date DESC
LIMIT 5
"""

df_verificacion = pd.read_sql(query, con=engine)

print("Consulta de prueba (Apple, últimos 5 registros en BD):")
display(df_verificacion)


print("\nEstructura de la tabla guardada:")
# Usar final_df (siempre definido: cargado desde SQL o creado en descarga)
df_schema = final_df.reset_index() if 'Date' not in final_df.columns else final_df
inspector = pd.io.sql.get_schema(df_schema, table_name)
print(inspector)

Consulta de prueba (Apple, últimos 5 registros en BD):


,Date,Ticker,Adj Close,RSI_14
0,2026-03-03 00:00:00.000000,AAPL,263.750000,41.597582
1,2026-03-02 00:00:00.000000,AAPL,264.720001,41.618711
2,2026-02-27 00:00:00.000000,AAPL,264.179993,38.924889
3,2026-02-26 00:00:00.000000,AAPL,272.950012,47.552395
4,2026-02-25 00:00:00.000000,AAPL,274.230011,48.163678



Estructura de la tabla guardada:
CREATE TABLE "sp500_daily_metrics" (
"Date" TEXT,
  "Ticker" TEXT,
  "Open" REAL,
  "High" REAL,
  "Low" REAL,
  "Close" REAL,
  "Adj Close" REAL,
  "Volume" REAL,
  "SMA_10" REAL,
  "SMA_50" REAL,
  "SMA_200" REAL,
  "EMA_12" REAL,
  "EMA_26" REAL,
  "Std_20" REAL,
  "Bollinger_Upper" REAL,
  "Bollinger_Lower" REAL,
  "Daily_Range" REAL,
  "Range_Pct" REAL,
  "Daily_Return" REAL,
  "Log_Return" REAL,
  "Cumulative_Return" REAL,
  "Is_Positive_Day" INTEGER,
  "MACD_Line" REAL,
  "MACD_Signal" REAL,
  "MACD_Hist" REAL,
  "RSI_14" REAL,
  "Momentum_10" REAL,
  "Vol_SMA_20" REAL,
  "Vol_Ratio" REAL,
  "Vol_Change" REAL,
  "High_Low_Spread" REAL,
  "Close_Open_Spread" REAL,
  "Typical_Price" REAL,
  "Z_Score_20" REAL,
  "Above_SMA200" INTEGER,
  "Pct_From_High_50" REAL
)


## Prueba

In [6]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Conectar a la base de datos que creamos
# (Asegúrate de que el nombre del archivo .db sea el mismo que usaste antes)
engine = create_engine('sqlite:///sp500_market_data.db')


query = """
SELECT * FROM sp500_daily_metrics
WHERE Ticker = 'AAPL'
ORDER BY Date DESC
LIMIT 5
"""


df_apple_recent = pd.read_sql(query, con=engine)


print("Mostrando los últimos 5 días de Apple:")
display(df_apple_recent)

Mostrando los últimos 5 días de Apple:


,Date,Ticker,Open,High,Low,Close,Adj Close,Volume,SMA_10,SMA_50,SMA_200,EMA_12,EMA_26,Std_20,Bollinger_Upper,Bollinger_Lower,Daily_Range,Range_Pct,Daily_Return,Log_Return,Cumulative_Return,Is_Positive_Day,MACD_Line,MACD_Signal,MACD_Hist,RSI_14,Momentum_10,Vol_SMA_20,Vol_Ratio,Vol_Change,High_Low_Spread,Close_Open_Spread,Typical_Price,Z_Score_20,Above_SMA200,Pct_From_High_50
0,2026-03-03 00:00:00.000000,AAPL,263.480011,265.559998,260.130005,263.750000,263.750000,37994695.0,266.766000,264.851136,243.020448,266.692592,266.137432,6.338108,277.527353,252.174920,5.429993,2.060875,-0.003664,-0.003671,1.519990,1,0.555160,0.908408,-0.353248,41.597582,-0.130005,49741509.75,0.763843,-0.091642,0.020874,0.001025,263.146667,-0.763806,1,-0.050781
1,2026-03-02 00:00:00.000000,AAPL,262.410004,266.529999,260.200012,264.720001,264.720001,41827900.0,266.779001,265.007854,242.760127,267.227609,266.328427,6.238171,277.484196,252.531512,6.329987,2.412250,0.002044,0.002042,1.525581,1,0.899182,0.996720,-0.097538,41.618711,8.940002,51537445.00,0.811602,-0.421999,0.024327,0.008803,263.816671,-0.668699,1,-0.047290
2,2026-02-27 00:00:00.000000,AAPL,272.809998,272.809998,262.890015,264.179993,264.179993,72366500.0,265.885001,265.200519,242.497947,267.683537,266.457101,6.544090,278.288700,252.112339,9.919983,3.636224,-0.032130,-0.032658,1.522468,0,1.226437,1.021104,0.205333,38.924889,2.449982,54068220.00,1.338429,1.237325,0.037734,-0.031634,266.626668,-0.678068,1,-0.049233
3,2026-02-26 00:00:00.000000,AAPL,274.950012,276.109985,270.799988,272.950012,272.950012,32345100.0,265.640002,265.393994,242.227800,268.320546,266.639269,6.897796,279.189586,251.598403,5.309998,1.931259,-0.004668,-0.004679,1.573010,0,1.681276,0.969771,0.711505,47.552395,-2.549988,53812545.00,0.601070,-0.040612,0.019609,-0.007274,273.286662,0.672643,1,-0.017671
4,2026-02-25 00:00:00.000000,AAPL,271.779999,274.940002,271.049988,274.230011,274.230011,33714300.0,265.895001,265.495391,241.851392,267.478824,266.134410,7.309362,280.114115,250.876667,3.890015,1.431310,0.007680,0.007651,1.580387,1,1.344415,0.791895,0.552520,48.163678,0.550018,54259690.00,0.621351,-0.282897,0.014352,0.009015,273.406667,0.924463,1,-0.013632


In [7]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('sqlite:///sp500_market_data.db')
table_name = 'sp500_daily_metrics'

query_filas = f"SELECT COUNT(*) FROM {table_name}"
df_count = pd.read_sql(query_filas, con=engine)
total_filas = df_count.iloc[0, 0]

query_cols = f"SELECT * FROM {table_name} LIMIT 1"
df_estructura = pd.read_sql(query_cols, con=engine)
total_columnas = len(df_estructura.columns)

total_datos = total_filas * total_columnas

print(f"Tabla: {table_name}")
print(f"Total de Filas: {total_filas:,.0f}")
print(f"Total de Columnas: {total_columnas:,.0f}")
print(f"Total de Datos: {total_datos:,.0f}")

Tabla: sp500_daily_metrics
Total de Filas: 151,258
Total de Columnas: 36
Total de Datos: 5,445,288


In [8]:
final_df

,Date,Ticker,Open,High,Low,Close,Adj Close,Volume,SMA_10,SMA_50,SMA_200,EMA_12,EMA_26,Std_20,Bollinger_Upper,Bollinger_Lower,Daily_Range,Range_Pct,Daily_Return,Log_Return,Cumulative_Return,Is_Positive_Day,MACD_Line,MACD_Signal,MACD_Hist,RSI_14,Momentum_10,Vol_SMA_20,Vol_Ratio,Vol_Change,High_Low_Spread,Close_Open_Spread,Typical_Price,Z_Score_20,Above_SMA200,Pct_From_High_50
0,2024-12-16 00:00:00.000000,MMM,129.899994,130.199997,128.860001,129.470001,126.503265,3097800.0,127.823310,127.854525,109.820000,127.553482,127.676281,1.886003,131.626531,124.082519,1.339996,1.031560,-0.003464,-0.003470,1.723489,0,-0.122799,0.110589,-0.233388,47.029955,-3.341644,2890235.00,1.071816,0.300504,0.010399,-0.003310,129.510000,-0.593711,1,-0.045408
1,2024-12-17 00:00:00.000000,MMM,128.410004,129.360001,127.650002,128.050003,125.115807,2693200.0,127.520415,127.735582,110.078582,127.178455,127.486616,1.966358,131.668299,123.802865,1.709999,1.331671,-0.010968,-0.011028,1.704586,0,-0.308161,0.026839,-0.335000,31.297554,-3.028954,2903245.00,0.927652,-0.130609,0.013396,-0.002804,128.353335,-1.222376,1,-0.055878
2,2024-12-18 00:00:00.000000,MMM,128.210007,129.369995,125.370003,125.529999,122.653549,3748900.0,127.039687,127.570504,110.319835,126.482316,127.128611,2.196472,131.963448,123.177559,3.999992,3.119875,-0.019680,-0.019876,1.671040,0,-0.646296,-0.107788,-0.538508,27.062569,-4.807274,2960665.00,1.266236,0.391987,0.031905,-0.020903,126.756666,-2.156155,1,-0.074458
3,2024-12-19 00:00:00.000000,MMM,126.650002,127.739998,125.709999,127.129997,124.216888,3038100.0,126.433895,127.430472,110.569587,126.133788,126.912928,2.242670,131.915811,122.945133,2.029999,1.602841,0.012746,0.012665,1.692339,1,-0.779140,-0.242058,-0.537081,29.617798,-6.057922,3027540.00,1.003488,-0.189602,0.016148,0.003790,126.859998,-1.399185,1,-0.062661
4,2024-12-20 00:00:00.000000,MMM,126.370003,130.350006,125.750000,129.279999,126.317619,8351300.0,126.059673,127.362584,110.829441,126.162070,126.868831,2.148907,131.660398,123.064770,4.600006,3.640109,0.016912,0.016770,1.720959,1,-0.706761,-0.334999,-0.371762,39.511845,-3.742226,3317865.00,2.517070,1.748856,0.036581,0.023028,128.460002,-0.527214,1,-0.046809
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151253,2026-02-25 00:00:00.000000,ZTS,128.699997,129.779999,127.019997,128.949997,128.949997,3024100.0,127.511000,125.150507,140.932399,127.399735,126.474103,2.292033,129.734573,120.566441,2.760002,2.144524,0.002254,0.002251,0.703829,1,0.925632,0.771072,0.154560,55.512003,0.819992,4812890.00,0.628333,-0.163296,0.021729,0.001943,128.583331,1.129781,0,-0.001394
151254,2026-02-26 00:00:00.000000,ZTS,129.100006,131.339996,128.440002,129.759995,129.759995,3637500.0,127.620000,125.382598,140.793620,127.762852,126.717502,2.160369,129.703335,121.061860,2.899994,2.246316,0.006281,0.006262,0.708250,1,1.045350,0.825927,0.219422,57.149580,1.089996,4688255.00,0.775875,0.202837,0.022579,0.005112,129.846664,1.392352,0,0.000000
151255,2026-02-27 00:00:00.000000,ZTS,128.929993,132.300003,128.250000,131.100006,131.100006,5546300.0,128.166000,125.595286,140.648437,128.276260,127.042132,2.024346,129.643978,121.546593,4.050003,3.141242,0.010327,0.010274,0.715564,1,1.234128,0.907568,0.326560,58.424922,5.460007,4729120.00,1.172797,0.524756,0.031579,0.016831,130.550003,1.916177,0,0.000000
151256,2026-03-02 00:00:00.000000,ZTS,130.050003,130.050003,127.089996,128.960007,128.960007,3063900.0,128.397001,125.720365,140.500020,128.381452,127.184197,1.977025,129.674415,121.766315,2.960007,2.276053,-0.016323,-0.016458,0.703884,0,1.197255,0.965505,0.231750,53.478631,2.310005,4619950.00,0.663189,-0.447578,0.023291,-0.008381,128.700002,0.774905,0,-0.016323
